In [ ]:
# 서울시 지하철역 엘리베이터 위치 정보

In [3]:
import requests               # 서울시 Open API 호출 (HTTP 요청용)
import time                   # API 호출 간격 조절 및 대기 시간 생성용
import polars as pl           # Pandas 대용, 고속 데이터프레임 처리 및 파케(Parquet) 저장용
from datetime import datetime # 수집 날짜/시간을 데이터에 기록하기 위한 타임스탬프용
from pathlib import Path      # 로컬 SSD 파일 경로(디렉토리 생성 및 관리) 제어용
import os
from sqlalchemy import create_engine, text
import sys

In [4]:
now = datetime.now()
dt_suffix = now.strftime("%Y%m%d")

## workspace/korea-public-data-pipeline-hub/01_seoul_subway_elevator/src/notebooks
current_dir = Path(".").resolve().as_posix()[3:]

print(current_dir)

ers/hongtaegyeong/workspace/korea-public-data-pipeline-hub/01_seoul_subway_elevator/src/notebooks


In [5]:
API_KEY = "7656714c4567757336356769446c6b" # KEY: 고유 인증키
SERVICE_NAME = "tbTraficElvtr"
START_INDEX = 1                            # START_INDEX: 페이징 시작 행 번호
END_INDEX = 1000                          # END_INDEX: 페이징 종료 행 번호 |
all_rows= []

##  http://openapi.seoul.go.kr:8088/(인증키)/xml/tbTraficElvtr/1/5/

In [7]:
if str(Path("../").resolve()) not in sys.path:
    sys.path.append(str(Path("../").resolve()))

# 2️⃣ [홍 대리 맞춤형 팩트 경로] src 폴더가 기준이 되었으니 그 아래 'ods'에서 바로 가져오면 끝!
from ods.SeoulOpenApiClient import SeoulOpenApiClient #

# 3️⃣ 인스턴스 생성 및 가동!
subway_api: SeoulOpenApiClient = SeoulOpenApiClient(API_KEY)
raw_data:dict  = subway_api.fetch_data(
    service_name=SERVICE_NAME,
    start_idx=START_INDEX,  # 💡 중간의 req_type은 건너뛰고 바로 여기로 배달!
    end_idx=END_INDEX
)

In [8]:
subway_status:dict = raw_data.get("tbTraficElvtr")

# 딕셔너리들이 담은 리스트로 변환)
subway_rows:list = subway_status.get("row")

In [9]:
len(subway_rows)

552

In [10]:
df: pl.DataFrame = pl.DataFrame(subway_rows)
print(type(df))

<class 'polars.dataframe.frame.DataFrame'>


In [ ]:
%%sql


In [11]:
df.write_parquet("../../../data_lake/seoul_subway/seoul_subway_elevator.parquet")

In [12]:
df_read = pl.read_parquet("../../../data_lake/seoul_subway/seoul_subway_elevator.parquet")
df_read.head(5)

NODE_TYPE,NODE_WKT,NODE_ID,NODE_TYPE_CD,SGG_CD,SGG_NM,EMD_CD,EMD_NM,SBWY_STN_CD,SBWY_STN_NM
str,str,f64,str,str,str,str,str,str,str
"""NODE""","""POINT(127.01744971746365 37.57…",212659.0,"""0""","""1111000000""","""종로구""","""1111017500""","""숭인동""","""268""","""동묘앞"""
"""NODE""","""POINT(127.01505874969273 37.57…",167351.0,"""1""","""1111000000""","""종로구""","""1111017400""","""창신동""","""270""","""창신"""
"""NODE""","""POINT(126.98515892357797 37.57…",211632.0,"""0""","""1111000000""","""종로구""","""1111013400""","""경운동""","""269""","""안국"""
"""NODE""","""POINT(127.01049296509703 37.57…",212410.0,"""0""","""1111000000""","""종로구""","""1111017400""","""창신동""","""272""","""동대문"""
"""NODE""","""POINT(127.02297735704515 37.57…",211950.0,"""0""","""1111000000""","""종로구""","""1111017500""","""숭인동""","""119""","""신설동"""


In [18]:
df_dim_lazy = df_read.lazy().with_columns([
    pl.col("NODE_ID").cast(pl.Int64).alias("node_id"),          # 노드 ID를 정수형(Int64)으로 변환 후 소문자로 표준화
    pl.col("NODE_WKT").cast(pl.String).alias("node_wkt"),        # GIS 공간 정보(WKT)를 문자열로 명확히 지정
    pl.col("NODE_TYPE_CD").cast(pl.String).alias("node_type_cd"),# 노드 타입 코드를 문자열로 변환
    pl.col("SGG_NM").cast(pl.String).alias("sgg_nm")             # 시군구 명칭을 문자열로 변환
])

SyntaxError: invalid syntax (518464224.py, line 2)